In [ ]:
%load_ext autoreload
%autoreload 2

Declaration of parameters (you must also add a tag for this cell - parameters)

In [ ]:
#2. specify parameters
pipeline_params={
}
step_params={
}
substep_params={   
}

In [ ]:
#3 define substep interface
from sinara.substep import NotebookSubstep, default_param_values, ENV_NAME, PIPELINE_NAME, ZONE_NAME, STEP_NAME, RUN_ID, ENTITY_NAME, ENTITY_PATH, SUBSTEP_NAME

substep = NotebookSubstep(pipeline_params, step_params, substep_params, **default_param_values("params/step_params.json"))

substep.interface(
    inputs =    
    [ 
      { STEP_NAME: "1_data_import", ENTITY_NAME: "test_coco_data"},
      { STEP_NAME: "1_data_import", ENTITY_NAME: "config"},
      { STEP_NAME: "2_model_train", ENTITY_NAME: "model"}      
    ],
    
    tmp_outputs =
    [
        { ENTITY_NAME: "cache_data" },
        { ENTITY_NAME: "cache_config" },
        { ENTITY_NAME: "model" }
    ]
)

substep.print_interface_info()

substep.exit_in_visualize_mode()

![interface 0_configure_eval_interface.drawio](./imgs/0_configure_eval_interface.drawio.png)

In [ ]:
#4 get substep.interface
inputs_data_import = substep.inputs(step_name = "1_data_import")
inputs_model_train = substep.inputs(step_name = "2_model_train")
tmp_outputs = substep.tmp_outputs()

print(f"{inputs_data_import.test_coco_data=}")
print(f"{inputs_data_import.config=}")
print(f"{inputs_model_train.model=}")

print(f"{tmp_outputs.cache_data=}")
print(f"{tmp_outputs.cache_config=}")
print(f"{tmp_outputs.model=}")

In [ ]:
# Create dir by URLs of tmp_outputs
import os
import os.path as osp
os.makedirs(tmp_outputs.cache_data, exist_ok=True)

In [ ]:
#5 run spark
from sinara.spark import SinaraSpark

spark = SinaraSpark.run_session(0)
SinaraSpark.ui_url()

#### Loading a trained model from the model_train component 
(weights, configs)

In [ ]:
import os.path as osp
import os
from sinara.store import SinaraStore
from pathlib import Path

# copy config from previos step to outputs
SinaraStore.copy_store_files_to_tmp(store_dir=inputs_data_import.config, tmp_dir=tmp_outputs.cache_config)
# Create SUCCESS file after copy from store
Path(osp.join(tmp_outputs.cache_config, '_SUCCESS')).touch()

In [ ]:
from sinara.store import SinaraStore
from pathlib import Path

# copy config from previos step to outputs
SinaraStore.copy_store_files_to_tmp(store_dir=inputs_model_train.model, tmp_dir=tmp_outputs.model)
# Create SUCCESS file after copy from store
Path(osp.join(tmp_outputs.model, '_SUCCESS')).touch()

In [ ]:
!ls -lh {tmp_outputs.model}

#### Loading test datasets (from step 1_data_import)

In [ ]:
### load dataset from parquet
def save_file(row): 
    total_dir_name = tmp_outputs.cache_data
    total_img_path = osp.join(total_dir_name, row.file_names)
    os.makedirs(total_dir_name, exist_ok=True)
    with open(total_img_path, 'wb') as f_id:
        f_id.write(row.files_binary)

In [ ]:
%%time

# CACHE TEST

print(f"spark read start")

df_spark = spark.read.parquet(inputs_data_import.test_coco_data)
df_spark.foreach(save_file)

In [ ]:
# Create SUCCESS file after copy from store
Path(osp.join(tmp_outputs.cache_data, '_SUCCESS')).touch()

In [ ]:
#9 stop spark
SinaraSpark.stop_session()